# 🏥 Healthcare AI Assistant
**Author:** Siva Ramakrishna | **Role:** AI Engineer — Generative AI & Machine Learning

---

## 📐 System Architecture

| Layer | Technology | Purpose |
|:---|:---|:---|
| **Document Ingestion** | PyMuPDF + pytesseract OCR | Parse PDF, TXT, CSV, MD files |
| **Chunking** | Overlapping 800-word windows (100-word overlap) | Preserve context across boundaries |
| **Embeddings** | `all-MiniLM-L6-v2` (SentenceTransformers) | Dense semantic vectors |
| **Retrieval** | BM25 lexical + Dense semantic → RRF Fusion | Hybrid best-of-both retrieval |
| **LLM Chain** | Gemini 2.5 Flash → Flash-Lite → Groq Llama-3.3 70B → Llama-3.1 8B → Sarvam AI | Multi-provider fallback |
| **Token Budget** | Dynamic classifier: `simple=300`, `complex=600`, `outscope=200` | Cost-efficient inference |
| **Output Schema** | Pydantic `ClinicalResponse` (JSON enforced) | Structured, auditable responses |
| **Agentic Tool** | `mock_check_available_slots(dept, date)` router | Intent-based workflow branching |
| **API** | FastAPI + Cloudflare Tunnel (public HTTPS) | Production-ready endpoints |

---

## 🔑 API Keys Required
Add these to **Colab Secrets** (🔑 left sidebar) before running:

| Secret Name | Provider | Get It At |
|:---|:---|:---|
| `GEMINI_API_KEY` | Google AI Studio | [aistudio.google.com](https://aistudio.google.com/app/apikey) |
| `GROQ_API_KEY` | Groq Console | [console.groq.com](https://console.groq.com/keys) |
| `SARVAM_API_KEY` | Sarvam AI *(optional)* | [dashboard.sarvam.ai](https://dashboard.sarvam.ai) |

---

## 🔁 Execution Order

> **Always run top-to-bottom on first launch.** After Cell 11 initialises the assistant, Cells 12–15 can be re-run independently.

| # | Cell Name | Purpose |
|:-:|:---|:---|
| 1 | Install Dependencies | pip install all packages + OCR system deps |
| 2 | API Key Setup | Load keys from Colab Secrets |
| 3 | Imports & Logging | All library imports + logger config |
| 4 | Token Budget & Query Classifier | Dynamic per-query tier classification |
| 5 | Pydantic Output Schema | Strict `ClinicalResponse` JSON enforcement |
| 6 | Synthetic Document Corpus | 7 base healthcare policy documents |
| 7 | Hybrid Retriever | BM25 + Dense → RRF fusion retriever |
| 8 | Multi-Provider LLM Chain | Fallback chain across 5 providers |
| 9 | Prompt Engineering & Message Builder | System prompt + context assembly |
| 10 | HealthcareAssistant Orchestrator | Main class: RAG + agentic routing |
| 11 | Initialise Assistant | Boot assistant (downloads model once, ~90MB) |
| 12 | Interactive Single Query | Test any single question |
| 13 | Batch Test — All Queries | Run all 11 test queries |
| 14 | Token Budget Audit | Pre-flight TPM cost check |
| 15 | FastAPI Server + Cloudflare Tunnel | Live public API via HTTPS |

---

## 📁 Synthetic Dataset

All documents are **fully synthetic** — no real patient data or PHI. Topics covered:
- `medication_refill_policy.txt` — Controlled substance rules, online portal workflow
- `telehealth_consultation_guidelines.txt` — Eligible services, technology requirements
- `insurance_eligibility_faq.txt` — Coverage, copays, parity laws
- `appointment_scheduling_policy.txt` — Booking channels, specialist referrals, no-show fees
- `hipaa_privacy_guidelines.txt` — Patient rights, PHI access, HHS complaint process
- `discharge_instructions.txt` — Activity restrictions, warning signs, follow-up care
- `Health_Data.txt` — WHO malaria global statistics (public data, synthetic format)

---

## 🔌 API Endpoints

| Method | Endpoint | Description |
|:---|:---|:---|
| `GET` | `/health` | Server health + corpus chunk count |
| `POST` | `/ingest` | Upload a PDF/TXT/CSV/MD file to extend the knowledge base |
| `POST` | `/ask` | Submit a natural-language question, receive a structured JSON answer |
| `GET` | `/docs` | Interactive Swagger UI |

---

## 🧠 Prompt Engineering Strategy

The system prompt enforces **strict grounding rules**:
- Answer only from retrieved context — never hallucinate
- Refuse to diagnose or give direct medical advice
- Return a fixed JSON schema: `analysis` (2 sentences) + `questions` (exactly 2) + `confidence_score`
- Identity exception: introduce itself when asked "who are you?"
- Out-of-scope queries → `confidence_score ≤ 0.2` + explicit limitation statement

```
SYSTEM_PROMPT (see Cell 9 for full text)
```

---

## ⚖️ Limitations & Future Improvements

| Limitation | Future Improvement |
|:---|:---|
| In-memory corpus (resets on restart) | Persist to ChromaDB / Pinecone |
| Mock appointment tool | Integrate real EHR scheduling API |
| English-only | Multilingual support via Sarvam AI |
| No auth on API | Add OAuth2 / API key middleware |
| Colab-only tunnel | Deploy to Cloud Run / Railway with custom domain |


## Cell 1 — Install Dependencies

Installs all Python packages and system-level OCR tools required for PDF parsing.

In [1]:
# Python packages
!pip install -q \
    langchain langchain-google-genai langchain-groq langchain-openai \
    rank_bm25 sentence-transformers pydantic python-dotenv \
    pymupdf python-multipart pytesseract pdf2image \
    uvicorn fastapi

# System packages for OCR (tesseract) and PDF rendering (poppler)
!apt-get install -y tesseract-ocr poppler-utils -q

print("✅ All dependencies installed successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 21.8 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (4,994 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118252 fil

## Cell 2 — API Key Setup

Loads API keys from **Colab Secrets** (recommended). Never hardcode secrets in notebooks you share.

In [2]:
import os

# ── Option A: Colab Secrets (recommended — keys never appear in code) ─────────
try:
    from google.colab import userdata

    KEY_MAP = {
        "GEMINI_API_KEY": "GOOGLE_API_KEY",
        "GROQ_API_KEY":   "GROQ_API_KEY",
        "SARVAM_API_KEY": "SARVAM_API_KEY",
    }
    for secret_name, env_name in KEY_MAP.items():
        try:
            val = userdata.get(secret_name)
            os.environ[env_name] = val
            masked = val[:8] + "*" * max(0, len(val) - 8)
            print(f"  ✅ {env_name:<22} loaded  ({masked})")
        except Exception:
            print(f"  ⚠️  {env_name:<22} NOT FOUND in Colab Secrets")

except ImportError:
    print("ℹ️  Not in Colab — using environment variables directly.")

# ── Option B: Local .env file (for local development) ─────────────────────────
# from dotenv import load_dotenv; load_dotenv()

# ── Option C: Temporary test values — DELETE before sharing ──────────────────
# os.environ["GOOGLE_API_KEY"] = "AIza..."
# os.environ["GROQ_API_KEY"]   = "gsk_..."
# os.environ["SARVAM_API_KEY"] = "sk-..."

active = [k for k in ["GOOGLE_API_KEY", "GROQ_API_KEY", "SARVAM_API_KEY"] if os.environ.get(k)]
print(f"\n🔑 Active providers: {active if active else 'NONE — set at least GOOGLE_API_KEY or GROQ_API_KEY!'}")


  ✅ GOOGLE_API_KEY         loaded  (AIzaSyBa*******************************)
  ✅ GROQ_API_KEY           loaded  (gsk_C7ZY************************************************)
  ✅ SARVAM_API_KEY         loaded  (sk_6lq8n****************************)

🔑 Active providers: ['GOOGLE_API_KEY', 'GROQ_API_KEY', 'SARVAM_API_KEY']


## Cell 3 — Imports & Logging

All library imports in one place. Logger is set to WARNING to suppress model download noise.

In [3]:
import json
import logging
import textwrap
import re
import io
import os
from typing import List, Optional

import numpy as np
from pydantic import BaseModel, Field
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# Configure logging — WARNING level suppresses model download progress bars
logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger("healthcare_ai")

print("✅ All imports successful.")


✅ All imports successful.


## Cell 4 — Token Budget & Dynamic Query Classifier

**Design rationale:** Instead of a flat token budget, queries are classified into three tiers:
- `simple` (300 tokens) — single-fact lookups, yes/no questions
- `complex` (600 tokens) — multi-point answers, policy lists, statistics
- `outscope` (200 tokens) — questions outside the corpus knowledge base

The classifier dynamically extracts vocabulary from the live `CORPUS` on every call,
so newly ingested documents (cancer, diabetes, etc.) are immediately queryable without code changes.


In [4]:
# ── Token budget tiers ───────────────────────────────────────────────────────
TOKEN_BUDGET = {
    "simple":   300,   # single-fact lookups
    "complex":  600,   # multi-point answers, policy lists, statistics
    "outscope": 200,   # questions outside the corpus knowledge base
}

# ── Signals that indicate a complex multi-point answer is needed ─────────────
_COMPLEX_SIGNALS = {
    "rights", "right", "hipaa", "discharge", "instructions", "policy",
    "list", "all", "what are", "what is the", "multiple", "steps",
    "guidelines", "rules", "requirements", "options", "procedures",
    "how many", "statistics", "data", "compare", "difference", "explain",
    "symptoms", "causes", "types", "stages", "effects", "treatment",
}

# ── Static base medical vocabulary (always present regardless of corpus) ──────
_BASE_MEDICAL_TERMS = {
    "patient", "medication", "refill", "telehealth", "hipaa", "insurance",
    "appointment", "doctor", "prescription", "discharge", "fever", "health",
    "record", "schedule", "cancel", "visit", "clinic", "symptom", "disease",
    "treatment", "diagnosis", "mortality", "infection", "vaccine", "deaths",
    "cases", "outbreak", "epidemic", "statistics", "report", "global", "who",
    "burden", "prevention", "resistance", "hospital", "surgery", "therapy",
    "cancer", "malaria", "diabetes", "cardiology", "neurology", "oncology",
}

# ── Stop words excluded from corpus vocabulary extraction ─────────────────────
_STOP_WORDS = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "are", "was", "were", "be", "been",
    "has", "have", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "this", "that", "these", "those", "it", "its",
    "as", "if", "then", "than", "so", "yet", "both", "each", "more", "most",
    "other", "into", "through", "during", "before", "after", "above", "such",
    "no", "not", "only", "same", "also", "just", "over", "under", "per",
    "up", "down", "out", "about", "between", "while", "where", "when",
    "which", "who", "whom", "how", "what", "why", "can", "their", "your",
    "our", "my", "his", "her", "we", "they", "you", "i", "he", "she",
}


def _get_corpus() -> list:
    """Safe CORPUS accessor — returns [] if Cell 6 hasn't run yet."""
    try:
        return CORPUS
    except NameError:
        return []


def _extract_corpus_terms(corpus: list) -> set:
    """
    Dynamically extract meaningful keywords from the live CORPUS.
    Called fresh on every classify_query() so newly ingested documents
    are immediately reflected without any code changes.
    """
    terms = set()
    for doc in corpus:
        text = doc.get("text", "").lower()
        for word in re.findall(r"[a-z]{4,}", text):
            if word not in _STOP_WORDS:
                terms.add(word)
    return terms


def classify_query(query: str) -> str:
    """
    Lightweight zero-cost classifier.
    Returns: 'simple' | 'complex' | 'outscope'
    """
    q_lower = query.lower()

    # Check for complex signals first (multi-point answers)
    if any(signal in q_lower for signal in _COMPLEX_SIGNALS):
        # Confirm at least one medical term is present in the query
        corpus_terms = _extract_corpus_terms(_get_corpus())
        all_terms    = _BASE_MEDICAL_TERMS | corpus_terms
        if any(term in q_lower for term in all_terms):
            return "complex"

    # Check if query touches any known medical domain
    corpus_terms = _extract_corpus_terms(_get_corpus())
    all_terms    = _BASE_MEDICAL_TERMS | corpus_terms
    if any(term in q_lower for term in all_terms):
        return "simple"

    return "outscope"


print("✅ Token budget tiers defined:")
for tier, tokens in TOKEN_BUDGET.items():
    print(f"   {tier:<10} → {tokens} output tokens")


✅ Token budget tiers defined:
   simple     → 300 output tokens
   complex    → 600 output tokens
   outscope   → 200 output tokens


## Cell 5 — Pydantic Output Schema

Strict schema enforcement ensures every LLM response is parseable, auditable,
and consistent regardless of which provider handles the request.


In [5]:
class ClinicalResponse(BaseModel):
    """
    Strict output schema for all Healthcare AI responses.
    Every field is validated — malformed LLM outputs are caught and retried.
    """
    analysis: str = Field(
        description="Brief 2-sentence clinical reasoning grounded in retrieved context."
    )
    questions: List[str] = Field(
        description="Exactly 2 concise follow-up questions for the patient.",
        min_length=2,
        max_length=2,
    )
    confidence_score: float = Field(
        description="Confidence score between 0.0 (no context) and 1.0 (strong context match).",
        ge=0.0,
        le=1.0,
    )


print("✅ ClinicalResponse schema defined.")
print(f"   Fields: {list(ClinicalResponse.model_fields.keys())}")
print()
print("   Expected JSON output shape:")
print('   {')
print('     "analysis": "<2-sentence answer>",')
print('     "questions": ["<Question 1>?", "<Question 2>?"],')
print('     "confidence_score": 0.0 to 1.0')
print('   }')


✅ ClinicalResponse schema defined.
   Fields: ['analysis', 'questions', 'confidence_score']

   Expected JSON output shape:
   {
     "analysis": "<2-sentence answer>",
     "questions": ["<Question 1>?", "<Question 2>?"],
     "confidence_score": 0.0 to 1.0
   }


## Cell 6 — Synthetic Document Corpus

Seven fully **synthetic** healthcare documents covering the topics required by the assignment.
No real patient data or PHI is used.

The `/ingest` API endpoint (Cell 15) extends this corpus at runtime with uploaded files.


In [6]:
CORPUS: List[dict] = [
    # ── 1. Medication Refill Policy ─────────────────────────────────────────
    {
        "source": "medication_refill_policy.txt",
        "text": (
            "MEDICATION REFILL POLICY — HOW TO REQUEST A REFILL: "
            "Patients may request prescription refills through: "
            "(1) Online patient portal at myhealth.example.com — allow 48-72 hours processing. "
            "(2) Phone: call 1-800-HEALTH-1 during business hours. "
            "(3) Telehealth follow-up: routine medications may be renewed if the prescribing "
            "physician determines it is clinically appropriate. "
            "IMPORTANT: Controlled substances (Schedule II-V) CANNOT be refilled via telehealth. "
            "An in-person visit with a DEA-registered provider is required for controlled substances. "
            "Early refill requests (before 75% of days supply used) will be denied by most insurers."
        ),
    },

    # ── 2. Telehealth Consultation Guidelines ────────────────────────────────
    {
        "source": "telehealth_consultation_guidelines.txt",
        "text": (
            "TELEHEALTH CONSULTATION GUIDELINES — ELIGIBLE SERVICES: "
            "Follow-up visits for established patients, mental health counselling, "
            "dermatology (photo-based), chronic disease management, and medication reviews. "
            "NOT eligible via telehealth: new patient physical exams, procedures requiring "
            "hands-on assessment, and controlled substance initial prescriptions. "
            "TECHNOLOGY REQUIREMENTS: smartphone, tablet, or PC with camera and microphone; "
            "stable internet connection (minimum 5 Mbps); MyHealth app or any HIPAA-compliant "
            "video platform such as Zoom for Healthcare or Doxy.me. "
            "INSURANCE: Telehealth parity laws in most states require equal coverage for "
            "telehealth and in-person visits. Separate pre-authorisation is NOT required if "
            "the service is already covered in-person."
        ),
    },

    # ── 3. Insurance Eligibility FAQ ─────────────────────────────────────────
    {
        "source": "insurance_eligibility_faq.txt",
        "text": (
            "INSURANCE ELIGIBILITY FAQ: "
            "Accepted plans: Medicare, Medicaid, most PPO and HMO plans, TRICARE, and CHIP. "
            "Telehealth parity laws require equal coverage for telehealth and in-person visits "
            "in most states. Contact your insurer to verify out-of-network benefits before booking. "
            "Copays for telehealth visits are typically the same as primary-care in-person copays. "
            "Uninsured patients may apply for a sliding-scale fee programme based on household income. "
            "Prior authorisation is required for specialist referrals, MRI/CT scans, and elective surgery. "
            "Coverage disputes: contact Member Services on the back of your insurance card."
        ),
    },

    # ── 4. Appointment Scheduling Policy ─────────────────────────────────────
    {
        "source": "appointment_scheduling_policy.txt",
        "text": (
            "APPOINTMENT SCHEDULING POLICY — BOOKING CHANNELS: "
            "Online: myhealth.example.com (available 24/7). "
            "Phone: 1-800-HEALTH-1 (Monday to Friday, 8 am to 6 pm). "
            "Walk-ins accepted for urgent care only — not for specialist consultations. "
            "SPECIALIST APPOINTMENTS (cardiology, neurology, oncology): require a PCP referral "
            "and may have a 2-4 week lead time depending on availability. "
            "CANCELLATION POLICY: Cancellations must be made at least 24 hours in advance "
            "to avoid a $50 no-show fee. Three consecutive no-shows may result in discharge "
            "from the practice. "
            "SAME-DAY URGENT SLOTS: released daily at 8 am — book early via the online portal."
        ),
    },

    # ── 5. HIPAA Privacy Guidelines ──────────────────────────────────────────
    {
        "source": "hipaa_privacy_guidelines.txt",
        "text": (
            "HIPAA PRIVACY GUIDELINES — PATIENT RIGHTS UNDER THE PRIVACY RULE: "
            "(1) ACCESS: Receive a copy of your medical records within 30 days of written request "
            "(electronic records: within 3 business days). "
            "(2) AMENDMENT: Request corrections or amendments to inaccurate records. "
            "(3) ACCOUNTING: Receive an accounting of all non-treatment disclosures of your PHI. "
            "(4) RESTRICTION: Request restrictions on certain uses and disclosures of your PHI. "
            "(5) NOTICE: Receive a paper copy of the Notice of Privacy Practices on request. "
            "(6) COMPLAINT: File a complaint with the HHS Office for Civil Rights at hhs.gov/ocr "
            "or call 1-800-368-1019 at no cost. "
            "Violations of HIPAA may result in civil penalties ranging from $100 to $50,000 per violation."
        ),
    },

    # ── 6. Patient Discharge Instructions ────────────────────────────────────
    {
        "source": "discharge_instructions.txt",
        "text": (
            "PATIENT DISCHARGE INSTRUCTIONS — PLEASE READ BEFORE LEAVING: "
            "1. ACTIVITY RESTRICTIONS: Avoid strenuous physical activity for at least 48 hours. "
            "Do not drive if you have received sedation or opioid pain medication. "
            "2. WARNING SIGNS — RETURN TO ER IMMEDIATELY IF YOU EXPERIENCE: "
            "chest pain or pressure, difficulty breathing, sudden severe headache, "
            "high fever above 38.5 degrees Celsius, wound redness or discharge, "
            "severe uncontrolled pain, or confusion or altered consciousness. "
            "3. MEDICATIONS: Take all prescribed medications exactly as directed. "
            "Do not skip doses. Do not stop antibiotics early even if you feel better. "
            "4. FOLLOW-UP APPOINTMENT: Schedule within 7-10 days of discharge. "
            "Call 1-800-HEALTH-1 or book online at myhealth.example.com. "
            "5. DIET: Follow any dietary restrictions given by your care team. "
            "Stay well hydrated — aim for 8 glasses of water per day unless fluid-restricted."
        ),
    },

    # ── 7. WHO Malaria Global Health Data (Synthetic format, public source) ──
    {
        "source": "global_health_data.txt",
        "text": (
            "WHO MALARIA GLOBAL STATISTICS — KEY FACTS (WHO World Malaria Report). "
            "DEATHS BY YEAR: 2022 = 608,000 deaths globally; 2021 = 627,000 deaths; "
            "2020 = 625,000 deaths; 2019 = 558,000 deaths. "
            "CASES BY YEAR: 2022 = 249 million cases worldwide; 2021 = 247 million cases. "
            "REGIONAL BURDEN: Sub-Saharan Africa accounts for over 94% of all malaria cases "
            "and deaths globally. Children under 5 represent 80% of all malaria fatalities "
            "in the African region. Nigeria, DRC, Uganda, Mozambique, and Angola account for "
            "over 50% of global malaria deaths. "
            "PARASITES: P. falciparum is responsible for 99% of African malaria deaths. "
            "P. vivax is the most geographically widespread species. "
            "TREATMENT: Artemisinin-based Combination Therapies (ACTs) are WHO first-line treatment. "
            "VACCINES: RTS,S/AS01 (Mosquirix) approved 2021, 30-40% efficacy. "
            "R21/Matrix-M approved 2023, up to 75% efficacy. "
            "ECONOMIC BURDEN: Malaria costs over $12 billion USD annually in lost productivity. "
            "WHO 2030 Target: 90% reduction in malaria incidence and mortality."
        ),
    },
]

print(f"✅ Synthetic corpus loaded: {len(CORPUS)} documents")
print()
for i, doc in enumerate(CORPUS, 1):
    print(f"   {i}. {doc['source']:<45} ({len(doc['text']):,} chars)")


✅ Synthetic corpus loaded: 7 documents

   1. medication_refill_policy.txt                  (630 chars)
   2. telehealth_consultation_guidelines.txt        (763 chars)
   3. insurance_eligibility_faq.txt                 (623 chars)
   4. appointment_scheduling_policy.txt             (627 chars)
   5. hipaa_privacy_guidelines.txt                  (728 chars)
   6. discharge_instructions.txt                    (894 chars)
   7. global_health_data.txt                        (1,017 chars)


## Cell 7 — Hybrid Retriever (BM25 + Dense → RRF Fusion)

**Why hybrid?**
- BM25 excels at exact keyword matches ("608,000 deaths", "Schedule II")
- Dense retrieval excels at semantic similarity ("chest pain after surgery" → discharge instructions)
- Reciprocal Rank Fusion (RRF) combines both rankings without needing score calibration


In [7]:
class HybridRetriever:
    """
    Combines BM25 lexical search and dense semantic search via
    Reciprocal Rank Fusion (RRF) for robust document retrieval.
    """

    RRF_K = 60  # Standard RRF constant — higher = smoother rank blending

    def __init__(self, corpus: List[dict], model_name: str = "all-MiniLM-L6-v2"):
        self.corpus = corpus
        texts = [c["text"] for c in corpus]

        # BM25 — lexical retrieval (exact keyword matching)
        tokenised  = [t.lower().split() for t in texts]
        self.bm25  = BM25Okapi(tokenised)

        # Dense — semantic retrieval (contextual similarity)
        print("⏳ Loading sentence-transformer model (downloads once, ~90 MB)...")
        self.encoder    = SentenceTransformer(model_name)
        self.embeddings = self.encoder.encode(texts, normalize_embeddings=True)
        print(f"✅ HybridRetriever ready — indexed {len(corpus)} chunks.")

    def _rrf(self, *rank_lists: List[int]) -> List[int]:
        """Merge multiple ranked lists using Reciprocal Rank Fusion."""
        scores: dict = {}
        for ranks in rank_lists:
            for rank, doc_idx in enumerate(ranks):
                scores[doc_idx] = scores.get(doc_idx, 0.0) + 1.0 / (self.RRF_K + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)

    def retrieve(self, query: str, top_k: int = 3) -> List[dict]:
        """Retrieve top_k most relevant chunks via BM25 + dense RRF fusion."""
        # BM25 ranking
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranks  = list(np.argsort(bm25_scores)[::-1])

        # Dense ranking
        q_emb        = self.encoder.encode([query], normalize_embeddings=True)
        dense_scores = (q_emb @ self.embeddings.T)[0]
        dense_ranks  = list(np.argsort(dense_scores)[::-1])

        # RRF fusion
        fused = self._rrf(bm25_ranks, dense_ranks)
        return [self.corpus[i] for i in fused[:top_k]]


print("✅ HybridRetriever class defined.")


✅ HybridRetriever class defined.


## Cell 8 — Multi-Provider LLM Fallback Chain

**Priority order for answer generation:**
1. **Gemini 2.5 Flash** — primary (best quality, structured output)
2. **Gemini 2.5 Flash-Lite** — secondary safety net
3. **Groq Llama-3.3 70B** — fallback 1 (fast, free tier)
4. **Groq Llama-3.1 8B** — fallback 2 (fastest, lowest latency)
5. **Sarvam AI sarvam-m** — fallback 3 (Indian healthcare context)

The chain is rebuilt per-query with the correct `max_tokens` for that tier,
ensuring cost efficiency without sacrificing answer quality.


In [8]:
PROVIDER_NAMES: dict = {}


def _build_sarvam_llm(max_tokens: int):
    """Build Sarvam AI LLM via OpenAI-compatible endpoint."""
    try:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(
            model="sarvam-m",
            api_key=os.environ.get("SARVAM_API_KEY", ""),
            base_url="https://api.sarvam.ai/v1",
            temperature=0.2,
            max_tokens=max_tokens,
        )
    except ImportError:
        logger.warning("langchain-openai not installed — Sarvam fallback unavailable.")
        return None


def build_reconstruction_chain():
    """
    Query rewrite chain (Groq only — fast and cheap for rewriting).
    Rewrites multi-turn queries into standalone questions for better retrieval.
    """
    providers = []
    global PROVIDER_NAMES

    if os.environ.get("GROQ_API_KEY"):
        for model, label in [
            ("llama-3.1-8b-instant",    "Groq Llama-3.1 8B (Rewrite)"),
            ("llama-3.3-70b-versatile", "Groq Llama-3.3 70B (Rewrite)"),
        ]:
            llm = ChatGroq(
                model=model,
                api_key=os.environ["GROQ_API_KEY"],
                temperature=0.0,
                max_tokens=150,
            )
            providers.append(llm)
            PROVIDER_NAMES[id(llm)] = label

    sarvam = _build_sarvam_llm(150)
    if sarvam:
        providers.append(sarvam)
        PROVIDER_NAMES[id(sarvam)] = "Sarvam AI (Rewrite)"

    if not providers:
        raise EnvironmentError("No rewrite providers — set GROQ_API_KEY.")

    primary, *fallbacks = providers
    chain = primary.with_fallbacks(fallbacks) if fallbacks else primary
    return chain, providers


def build_answer_chain(max_tokens: int = 300):
    """
    Answer generation chain with multi-provider fallback.
    Priority: Gemini 2.5 Flash → Flash-Lite → Groq 70B → Groq 8B → Sarvam AI.
    """
    providers = []
    global PROVIDER_NAMES

    if os.environ.get("GOOGLE_API_KEY"):
        for model, label in [
            ("gemini-2.5-flash",      "Gemini 2.5 Flash"),
            ("gemini-2.5-flash-lite", "Gemini 2.5 Flash-Lite"),
        ]:
            llm = ChatGoogleGenerativeAI(
                model=model,
                google_api_key=os.environ["GOOGLE_API_KEY"],
                temperature=0.2,
                max_output_tokens=max_tokens,
            )
            providers.append(llm)
            PROVIDER_NAMES[id(llm)] = label

    if os.environ.get("GROQ_API_KEY"):
        for model, label in [
            ("llama-3.3-70b-versatile", "Groq Llama-3.3 70B"),
            ("llama-3.1-8b-instant",    "Groq Llama-3.1 8B"),
        ]:
            llm = ChatGroq(
                model=model,
                api_key=os.environ["GROQ_API_KEY"],
                temperature=0.2,
                max_tokens=max_tokens,
            )
            providers.append(llm)
            PROVIDER_NAMES[id(llm)] = label

    sarvam = _build_sarvam_llm(max_tokens)
    if sarvam:
        providers.append(sarvam)
        PROVIDER_NAMES[id(sarvam)] = "Sarvam AI"

    if not providers:
        raise EnvironmentError("No answer providers — set GOOGLE_API_KEY or GROQ_API_KEY.")

    primary, *fallbacks = providers
    chain = primary.with_fallbacks(fallbacks) if fallbacks else primary
    return chain, providers


print("✅ Multi-provider fallback chain builders defined.")
print("   Answer priority: Gemini 2.5 Flash → Flash-Lite → Groq 70B → Groq 8B → Sarvam")


✅ Multi-provider fallback chain builders defined.
   Answer priority: Gemini 2.5 Flash → Flash-Lite → Groq 70B → Groq 8B → Sarvam


## Cell 9 — System Prompt & Message Builder

**Prompt Engineering Strategy:**

The system prompt enforces four key constraints:
1. **Context-only answers** — the LLM is explicitly told to use ONLY the retrieved chunks
2. **No hallucination** — "Never invent clinical facts not present in the context"
3. **Safe responses** — no direct medical diagnosis or unsafe advice
4. **Structured output** — strict JSON schema with exactly 2 sentences + 2 questions

`MAX_CHUNK_CHARS = 1500` ensures statistics buried deep in long documents (e.g., death counts,
percentages) are fully visible to the LLM — previously capped at 400 which caused missed answers.


In [9]:
# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = textwrap.dedent("""
    You are a Healthcare AI Assistant built for Mindbowser. Your role is to help
    patients and staff understand clinic policies and healthcare information.

    Use ONLY the retrieved context provided below. Respond with a single JSON object —
    no markdown, no code fences, no extra text — matching this exact schema:

    {
      "analysis": "<2-sentence clinical reasoning based on retrieved context>",
      "questions": ["<Follow-up Question 1>?", "<Follow-up Question 2>?"],
      "confidence_score": <float between 0.0 and 1.0>
    }

    STRICT RULES:
    - "questions" must contain EXACTLY 2 items — no more, no less.
    - "analysis" must be exactly 2 sentences maximum.
    - NEVER invent clinical facts not present in the retrieved context.
    - NEVER provide direct medical diagnosis or specific dosage recommendations.
    - NEVER recommend specific treatments — refer patients to their care team.
    - If information is NOT in the context: set confidence_score <= 0.2 and state
      the limitation clearly in "analysis"; still provide 2 clarifying questions.
    - IDENTITY: If asked who you are, introduce yourself as the Healthcare AI
      Assistant powered by {CURRENT_MODEL} and set confidence_score to 1.0.
    - Do NOT include thinking tags, reasoning blocks, or <think> content in output.
""").strip()

# ── Query Rewrite Prompt ───────────────────────────────────────────────────────
REWRITE_PROMPT = textwrap.dedent("""
    Given the following conversation history and the user's latest query, rewrite the
    latest query into a clear, standalone question containing all necessary context.
    If the query is already standalone, return it exactly as-is.
    Do NOT answer the question — ONLY return the rewritten standalone text.
""").strip()

# ── Configuration ──────────────────────────────────────────────────────────────
MAX_HISTORY_TURNS = 2     # Number of past turns included in context window
MAX_CHUNK_CHARS   = 1500  # Characters per chunk sent to LLM (raised from 400 to capture statistics)


def strip_think_tags(text: str) -> str:
    """Remove <think>...</think> blocks and extract the JSON object."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    brace_pos = text.find("{")
    if brace_pos != -1:
        text = text[brace_pos:]
    return text.strip().lstrip("```json").lstrip("```").rstrip("```").strip()


def build_rewrite_messages(query: str, history: list) -> list:
    """Build message list for query rewriting with conversation history."""
    messages = [SystemMessage(content=REWRITE_PROMPT)]
    for msg in history[-(MAX_HISTORY_TURNS * 2):]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))
    messages.append(HumanMessage(content=f"LATEST QUERY TO REWRITE: {query}"))
    return messages


def build_messages(query: str, chunks: list, history: list, current_model: str = "an AI model") -> list:
    """Build the full message list for answer generation."""
    context_block = "\n\n".join(
        f"[Source: {c['source']}]\n{c['text'][:MAX_CHUNK_CHARS]}"
        for c in chunks
    )
    prompt = SYSTEM_PROMPT.replace("{CURRENT_MODEL}", current_model)
    messages = [SystemMessage(content=prompt)]
    for msg in history[-(MAX_HISTORY_TURNS * 2):]:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))
    messages.append(
        HumanMessage(content=f"RETRIEVED CONTEXT:\n{context_block}\n\nPATIENT QUERY: {query}")
    )
    return messages


print("✅ System prompt and message builders defined.")
print(f"   MAX_HISTORY_TURNS = {MAX_HISTORY_TURNS}")
print(f"   MAX_CHUNK_CHARS   = {MAX_CHUNK_CHARS}")


✅ System prompt and message builders defined.
   MAX_HISTORY_TURNS = 2
   MAX_CHUNK_CHARS   = 1500


## Cell 10 — HealthcareAssistant Orchestrator

The main class that ties together:
- **Agentic routing** — detects appointment booking intent and bypasses RAG
- **Query rewriting** — multi-turn conversation context handling
- **RAG pipeline** — hybrid retrieval → LLM answer generation
- **Fallback handling** — graceful degradation if all providers fail


In [10]:
def clean_and_parse_json(raw_text: str) -> dict:
    """
    Robustly extracts and parses a JSON object from raw LLM output.
    Handles single-quote JSON, markdown code fences, and think tags.
    """
    text = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL | re.IGNORECASE)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in LLM response.")
    j = match.group(0)
    j = re.sub(r"'\s*:\s*",   '": ', j)
    j = re.sub(r"{\s*'",        '{"',  j)
    j = re.sub(r"'\s*,\s*'",  '", "', j)
    j = re.sub(r"'\s*,\s*\"", '", "', j)
    j = re.sub(r"\"\s*,\s*'", '", "', j)
    j = re.sub(r"'\s*\]",     '"]',  j)
    j = re.sub(r"\[\s*'",     '["',  j)
    return json.loads(j)


class HealthcareAssistant:
    """
    Main Healthcare AI orchestrator.
    Implements agentic routing, query rewriting, hybrid RAG, and multi-provider LLM fallback.
    """

    def __init__(self, corpus: list):
        self.retriever = HybridRetriever(corpus)
        self.history: list = []

    # ── Agentic Tool: Mock Appointment Booking ─────────────────────────────────
    def mock_check_available_slots(self, department: str, date: str) -> dict:
        """
        Mock agentic tool for appointment slot checking.
        In production, this would call an EHR scheduling API.
        Demonstrates the Mindbowser agentic workflow requirement.
        """
        return {
            "analysis": (
                f"I checked mock availability for {department} on {date}. "
                "Specialist appointments require a PCP referral and have a 2-4 week "
                "lead time, but urgent slots are released daily at 8 am."
            ),
            "questions": [
                "Would you like me to note your PCP details to verify referral status?",
                "Do you want to check general practitioner availability instead?",
            ],
            "confidence_score": 1.0,
        }

    # ── Main Ask Method ───────────────────────────────────────────────────────
    def ask(self, query: str, top_k: int = 3) -> dict:
        """
        Process a user query through the full pipeline:
        1. Classify query tier (simple / complex / outscope)
        2. Route to agentic tool OR RAG pipeline
        3. Rewrite query for multi-turn context (if history exists)
        4. Retrieve relevant chunks via HybridRetriever
        5. Generate structured answer via LLM fallback chain
        6. Update conversation history
        """
        q_lower = query.lower()
        tier    = classify_query(query)
        budget  = TOKEN_BUDGET[tier]

        # ── Step 1: Agentic intent routing (bypass RAG for appointment booking) ──
        if "book" in q_lower or "appointment" in q_lower or "slot" in q_lower:
            dept   = "Cardiology" if "cardiology" in q_lower else "General Medicine"
            date   = "Monday"     if "monday"     in q_lower else "your requested date"
            parsed = self.mock_check_available_slots(dept, date)
            llm_used            = "Agentic Mock Router Tool"
            chunks              = []
            reconstructed_query = query

        else:
            # ── Step 2: Query rewriting for multi-turn conversations ─────────────
            reconstructed_query = query
            if self.history:
                _, rw_providers = build_reconstruction_chain()
                rw_msgs = build_rewrite_messages(query, self.history)
                for provider in rw_providers:
                    try:
                        res = provider.invoke(rw_msgs)
                        reconstructed_query = res.content.strip()
                        break
                    except Exception:
                        continue

            # ── Step 3: Hybrid retrieval ─────────────────────────────────────────
            chunks           = self.retriever.retrieve(reconstructed_query, top_k=top_k)
            _, ans_providers = build_answer_chain(max_tokens=budget)
            parsed           = None
            llm_used         = "unknown"

            # ── Step 4: LLM answer generation with fallback chain ────────────────
            for provider in ans_providers:
                label = PROVIDER_NAMES.get(id(provider), "unknown")
                try:
                    messages = build_messages(query, chunks, self.history, current_model=label)
                    response = provider.invoke(messages)
                    parsed   = clean_and_parse_json(response.content)
                    if "analysis" not in parsed or "questions" not in parsed:
                        raise KeyError("Required schema keys missing.")
                    llm_used = label
                    break
                except Exception as exc:
                    logger.warning(f"[{label}] failed: {str(exc)}")
                    continue

        # ── Step 5: Hard fallback if ALL providers fail ──────────────────────────
        if parsed is None:
            parsed = {
                "analysis": "All AI providers failed to generate a valid response. Please try again.",
                "questions": ["Can you describe your symptoms in more detail?", "When did this start?"],
                "confidence_score": 0.0,
            }
            llm_used = "none"

        # ── Step 6: Out-of-scope override ────────────────────────────────────────
        # Only applied for TRUE out-of-corpus queries (not low-confidence real answers)
        if tier == "outscope":
            parsed["analysis"] = "I could not find this information in the provided documents."

        # ── Step 7: Build result object ──────────────────────────────────────────
        # FIX: store full {document, chunk} objects instead of just filenames
        result = dict(parsed)
        result["sources"] = [
            {"document": c["source"], "chunk": c["text"][:300]}
            for c in chunks
        ] if chunks else [{"document": "Mock Tool Engine", "chunk": ""}]
        result["llm_used"]         = llm_used
        result["token_tier"]       = tier
        result["budget_used"]      = budget
        result["rewritten_query"]  = reconstructed_query

        # ── Step 8: Update conversation history ──────────────────────────────────
        self.history.append({"role": "user",      "content": query})
        self.history.append({"role": "assistant",  "content": json.dumps(result)})
        return result


# ── Pretty Printer ─────────────────────────────────────────────────────────────
def confidence_icon(score: float) -> str:
    if score >= 0.8: return "🟢 HIGH"
    if score >= 0.5: return "🟡 MEDIUM"
    if score >= 0.2: return "🟠 LOW"
    return "🔴 VERY LOW"

def print_result(q_num: int, query: str, result: dict):
    sep = "=" * 72
    print(f"\n{sep}")
    print(f"  [{q_num:02d}] {query}")
    rw_q = result.get("rewritten_query", query)
    if rw_q != query:
        print(f"       ↳ Rewritten: {rw_q}")
    print(f"{'-' * 72}")
    print(f"  📋 Analysis:")
    print(f"     {result.get('analysis', 'N/A')}")
    print(f"\n  ❓ Follow-up Questions:")
    for i, q in enumerate(result.get("questions", []), 1):
        print(f"     {i}. {q}")
    score = result.get("confidence_score", 0.0)
    print(f"\n  {confidence_icon(score)} Confidence  : {score:.2f}")
    print(f"  🤖 LLM Used     : {result.get('llm_used', '?')}")
    print(f"  📊 Token Tier   : {result.get('token_tier', '?')} ({result.get('budget_used', '?')} tokens)")
    # FIX: sources are now dicts, so extract document names for display
    sources = result.get("sources", [])
    source_names = [s["document"] if isinstance(s, dict) else s for s in sources]
    print(f"  📄 Sources      : {', '.join(source_names)}")
    print(sep)


print("✅ HealthcareAssistant orchestrator loaded.")

✅ HealthcareAssistant orchestrator loaded.


## Cell 11 — Initialise Assistant

> ⚠️ Downloads `all-MiniLM-L6-v2` (~90 MB) on first run. The model is cached afterward.
> This cell rebuilds the retriever index over all CORPUS chunks.


In [11]:
assistant     = HealthcareAssistant(CORPUS)
API_ASSISTANT = assistant  # shared reference used by the FastAPI server (Cell 15)

print(f"\n✅ Healthcare AI Assistant initialised.")
print(f"   Corpus chunks indexed : {len(CORPUS)}")
print(f"   Conversation history  : {len(assistant.history)} messages")


⏳ Loading sentence-transformer model (downloads once, ~90 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ HybridRetriever ready — indexed 7 chunks.

✅ Healthcare AI Assistant initialised.
   Corpus chunks indexed : 7
   Conversation history  : 0 messages


## Cell 12 — Interactive Single Query

Change `my_query` to any question and re-run this cell to test.

In [12]:
# ── Change this to test any question ─────────────────────────────────────────
my_query = "Can a patient request a medication refill through telehealth?"

result = assistant.ask(my_query)
print_result(0, my_query, result)
print("\nRaw JSON output:")
print(json.dumps(result, indent=2))



  [00] Can a patient request a medication refill through telehealth?
------------------------------------------------------------------------
  📋 Analysis:
     Patients can request refills for routine medications through telehealth follow-up if the prescribing physician deems it clinically appropriate. However, controlled substances (Schedule II-V) cannot be refilled via telehealth and require an in-person visit with a DEA-registered provider.

  ❓ Follow-up Questions:
     1. What is the process for requesting a refill for a controlled substance?
     2. How long does it typically take to process a medication refill request submitted through the online patient portal?

  🟢 HIGH Confidence  : 1.00
  🤖 LLM Used     : Gemini 2.5 Flash
  📊 Token Tier   : simple (300 tokens)
  📄 Sources      : medication_refill_policy.txt, telehealth_consultation_guidelines.txt, hipaa_privacy_guidelines.txt

Raw JSON output:
{
  "analysis": "Patients can request refills for routine medications through te

## Cell 13 — Batch Test — All Queries

Runs all 11 test queries covering:
- In-corpus policy questions
- Agentic appointment routing
- Global health statistics
- Out-of-corpus edge cases
- Identity question


In [13]:
# TEST_QUERIES = [
#     # ── In-corpus: Policy questions ───────────────────────────────────────────
#     "Can a patient request a medication refill through telehealth?",
#     "What are my rights under HIPAA for accessing medical records?",
#     "What should I do if I have a fever after discharge from hospital?",
#     "Does telehealth require separate insurance authorization?",
#     "What technology do I need for a telehealth consultation?",
#     "Can controlled substances be refilled via telehealth?",
#     "What is the cancellation policy for appointments?",
#     # ── Agentic: Appointment booking (bypasses RAG → mock tool) ──────────────
#     "Can I book a cardiology appointment for Monday?",
#     # ── In-corpus: Global health statistics ──────────────────────────────────
#     "How many malaria deaths occurred in 2022?",
#     # ── Out-of-corpus: Edge cases ─────────────────────────────────────────────
#     "What is the stock price of Apple?",
#     # ── Identity ──────────────────────────────────────────────────────────────
#     "Who are you?",
# ]

# print("=" * 72)
# print("  Healthcare AI Assistant — Batch Test")
# print(f"  Corpus: {len(CORPUS)} indexed chunks | Queries: {len(TEST_QUERIES)}")
# print("=" * 72)

# for idx, query in enumerate(TEST_QUERIES, 1):
#     result = assistant.ask(query)
#     print_result(idx, query, result)


## Cell 14 — Token Budget Audit

Pre-flight check to verify total token usage stays within Groq's free-tier TPM limits
before running a large batch. Useful for production capacity planning.


In [14]:
# AUDIT_QUERIES = [
#     "Can a patient request a medication refill through telehealth?",
#     "What are my rights under HIPAA for accessing medical records?",
#     "What should I do if I have a fever after discharge from hospital?",
#     "Does telehealth require separate insurance authorization?",
#     "Can I book a cardiology appointment for Monday?",
#     "How many malaria deaths occurred in 2022?",
#     "What is the stock price of Apple?",
# ]

# TIER_ICONS = {"simple": "🔵 SIMPLE", "complex": "🟣 COMPLEX", "outscope": "⚪ OUTSCOPE"}

# print(f"{'#':<3} {'Query':<55} {'Tier':<10} {'Budget':>7}")
# print("-" * 80)
# for i, q in enumerate(AUDIT_QUERIES, 1):
#     tier   = classify_query(q)
#     budget = TOKEN_BUDGET[tier]
#     icon   = TIER_ICONS[tier]
#     print(f"{i:<3} {q[:53]:<55} {icon:<18} {budget:>5} tok")

# total = sum(TOKEN_BUDGET[classify_query(q)] for q in AUDIT_QUERIES)
# print("-" * 80)
# print(f"{'Total reserved output tokens':<68} {total:>5}")
# print(f"{'Groq free tier capacity (~6,000 TPM per model)':<68} {'OK ✅' if total <= 18000 else 'EXCEEDS ⚠️':>5}")


## Cell 15 — FastAPI Server + Cloudflare Tunnel

Starts the production API server with three endpoints:
- `GET /health` — server health + corpus chunk count
- `POST /ingest` — upload a PDF/TXT/CSV/MD file to extend the knowledge base
- `POST /ask` — submit a question, receive a structured JSON answer
- `GET /docs` — interactive Swagger UI

The Cloudflare tunnel exposes the server publicly over HTTPS without port forwarding.

> 💡 **Kill switch:** If you get "address already in use", uncomment and run the `!fuser -k 8000/tcp` cell below first.


In [15]:
import threading
import uvicorn
import fitz
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel as PydanticBase
from typing import List

app = FastAPI(
    title="Mindbowser Hackathon — Healthcare AI API",
    description=(
        "Production RAG API — Hybrid BM25 + Dense retrieval "
        "with multi-provider LLM fallback. No real patient data used."
    ),
    version="4.2",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryPayload(PydanticBase):
    question: str


# ── Text Chunker ───────────────────────────────────────────────────────────────
def chunk_text(source: str, text: str, chunk_size: int = 800, overlap: int = 100) -> list:
    words  = text.split()
    chunks = []
    step   = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i: i + chunk_size])
        if len(chunk.strip()) >= 50:
            chunks.append({"source": source, "text": chunk})
    return chunks


# ── GET /health ────────────────────────────────────────────────────────────────
@app.get("/health")
def health_endpoint():
    """Returns server health status and corpus chunk count."""
    return {
        "status":            "healthy",
        "version":           "4.2",
        "providers_indexed": len(PROVIDER_NAMES),
        "corpus_chunks":     len(CORPUS),
    }


# ── POST /ingest ───────────────────────────────────────────────────────────────
@app.post("/ingest")
async def ingest_endpoint(file: UploadFile = File(...)):
    """Ingest a document into the knowledge base. Supported: PDF, TXT, CSV, MD."""
    global API_ASSISTANT
    try:
        content_bytes = await file.read()
        content_text  = ""
        filename      = file.filename.lower()

        if filename.endswith(".pdf"):
            try:
                doc = fitz.open(stream=content_bytes, filetype="pdf")
                for page in doc:
                    extracted = page.get_text()
                    if extracted:
                        content_text += extracted + "\n"
                doc.close()
            except Exception as pdf_err:
                return JSONResponse(
                    status_code=500,
                    content={"error": f"PDF parsing failed: {str(pdf_err)}"}
                )

            if not content_text.strip():
                try:
                    import pytesseract
                    from pdf2image import convert_from_bytes
                    images = convert_from_bytes(content_bytes, dpi=300)
                    for img in images:
                        ocr_text = pytesseract.image_to_string(img)
                        if ocr_text:
                            content_text += ocr_text + "\n"
                except ImportError:
                    return JSONResponse(
                        status_code=422,
                        content={
                            "message": (
                                "PDF has no text layer and OCR libraries are not installed. "
                                "Run: !pip install pytesseract pdf2image && "
                                "!apt-get install -y tesseract-ocr poppler-utils"
                            ),
                            "total_corpus_chunks": len(CORPUS),
                        }
                    )

        elif filename.endswith((".txt", ".csv", ".md")):
            try:
                content_text = content_bytes.decode("utf-8")
            except UnicodeDecodeError:
                content_text = content_bytes.decode("latin-1")
        else:
            return {
                "message": f"Skipped '{file.filename}'. Accepted: .pdf .txt .csv .md",
                "total_corpus_chunks": len(CORPUS),
            }

        if not content_text.strip():
            return JSONResponse(
                status_code=422,
                content={
                    "message": f"No readable text extracted from '{file.filename}'.",
                    "total_corpus_chunks": len(CORPUS),
                }
            )

        before    = len(CORPUS)
        CORPUS[:] = [c for c in CORPUS if c["source"] != file.filename]
        removed   = before - len(CORPUS)
        if removed:
            print(f"Replaced {removed} old chunks for '{file.filename}'")

        new_chunks = chunk_text(file.filename, content_text, chunk_size=800, overlap=100)
        CORPUS.extend(new_chunks)
        API_ASSISTANT = HealthcareAssistant(CORPUS)

        return {
            "message": (
                f"Successfully ingested '{file.filename}'. "
                f"Extracted {len(content_text):,} characters → {len(new_chunks)} chunks."
            ),
            "chunks_created":      len(new_chunks),
            "total_corpus_chunks": len(CORPUS),
        }

    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})


# ── POST /ask ──────────────────────────────────────────────────────────────────
@app.post("/ask")
def ask_endpoint(payload: QueryPayload):
    """Submit a single natural-language question."""
    try:
        raw = API_ASSISTANT.ask(payload.question)

        # FIX: sources now carry real chunk text from Cell 10 — pass through directly
        # FIX: confidence now includes "medium" tier as required by assignment spec
        score = raw["confidence_score"]
        if score >= 0.7:
            confidence_label = "high"
        elif score >= 0.4:
            confidence_label = "medium"
        else:
            confidence_label = "low"

        return {
            "answer":              raw["analysis"],
            "sources":             raw.get("sources", []),
            "confidence":          confidence_label,
            "follow_up_questions": raw.get("questions", []),
            "metadata": {
                "llm_used":          raw["llm_used"],
                "token_tier":        raw["token_tier"],
                "token_budget_used": raw["budget_used"],
            },
        }
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})


# ── Start server ───────────────────────────────────────────────────────────────
def run_api_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

api_thread = threading.Thread(target=run_api_server, daemon=True)
api_thread.start()
print("✅ Healthcare AI API online at http://127.0.0.1:8000")
print("   Swagger UI : http://127.0.0.1:8000/docs")

✅ Healthcare AI API online at http://127.0.0.1:8000
   Swagger UI : http://127.0.0.1:8000/docs


In [16]:
import os, time, subprocess

print("Downloading Cloudflare Tunnel binary...")
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
os.system("rm cloudflared-linux-amd64.deb")

print("Launching Cloudflare Tunnel on port 8000...")
tunnel_log  = open("cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log, stderr=tunnel_log
)
time.sleep(6)

public_url = None
with open("cloudflared.log", "r") as f:
    for line in f:
        if "trycloudflare.com" in line:
            for part in line.split():
                if "trycloudflare.com" in part:
                    public_url = part.strip()
                    break

if public_url:
    ask_body = '\'{"question": "Can I refill medication via telehealth?"}\''

    print()
    print("=" * 72)
    print("  MINDBOWSER HACKATHON — LIVE PUBLIC API")
    print("=" * 72)
    print(f"  BASE URL : {public_url}")
    print(f"  HEALTH   : {public_url}/health")
    print(f"  ASK      : {public_url}/ask")
    print(f"  INGEST   : {public_url}/ingest")
    print(f"  SWAGGER  : {public_url}/docs")
    print("=" * 72)
    print()
    print("  Paste the BASE URL into Swagger /docs to test all endpoints.")
    print()
    print(f"  curl {public_url}/health")
    print(f"  curl -X POST {public_url}/ask -H \"Content-Type: application/json\" -d {ask_body}")
else:
    print("Failed to get tunnel URL. Check cloudflared.log:")
    !cat cloudflared.log

Launching Cloudflare Tunnel on port 8000...

  MINDBOWSER HACKATHON — LIVE PUBLIC API
  BASE URL : https://illustrations-languages-commissioners-nominations.trycloudflare.com
  HEALTH   : https://illustrations-languages-commissioners-nominations.trycloudflare.com/health
  ASK      : https://illustrations-languages-commissioners-nominations.trycloudflare.com/ask
  INGEST   : https://illustrations-languages-commissioners-nominations.trycloudflare.com/ingest
  SWAGGER  : https://illustrations-languages-commissioners-nominations.trycloudflare.com/docs

  Paste the BASE URL into Swagger /docs to test all endpoints.

  curl https://illustrations-languages-commissioners-nominations.trycloudflare.com/health
  curl -X POST https://illustrations-languages-commissioners-nominations.trycloudflare.com/ask -H "Content-Type: application/json" -d '{"question": "Can I refill medication via telehealth?"}'


In [17]:
# {
#   "questions": [
#     "1. What is the current status of progress on Universal Health Coverage (UHC) according to the report? How many additional people gained access to essential health services without financial hardship by 2024?",
# "2. According to the key messages, how many people were pushed into extreme poverty due to out-of-pocket health spending in 2019?",
# "3. What percentage of the global population spent more than 10% of their household budget on out-of-pocket health payments?"

#   ]
# }

In [18]:
# # # ── Kill switch — uncomment if port 8000 is already in use ──────────────────
# !fuser -k 8000/tcp